# 68. 같은 검색 결과에서 구조화 사실 표현과 답변 지시문 비교

개발 평가: 67번에서 선택한 검색 설정 하나로 카드·순위·기존 비구조화 근거를 고정한다. A는 기존 근거, B는 같은 근거에 DB 사실·관계를 JSON으로 추가, C는 B와 정확히 같은 내용을 한글 필드 표현으로 추가한다. 기존/직접질문 중심 지시문을 완전 교차한다.

12질문 × 2프로필 × 3입력 × 2지시문 × 3회 = 432논리답변. 원문·운영 코드·각 최적화 프로필은 읽기 전용이다. 수동 정답표는 자동 사실 조립기의 입력이 아니다. 묻지 않은 조건의 단순 생략은 직접 답변 실패가 아니며 실제 주장 오류는 실패다.

재사용 helper는 69번에서도 같은 실행 계약을 지키기 위해 분리했다. `68_fact_input.py`는 동일 내용/관계의 두 표현과 입력 상한을 검증한다. `68_69_retrieval_runtime.py`는 운영 검색을 실험 사본에 연결한다. `68_69_answer_runtime.py`는 실제 운영 답변 생성·근거 소유권 복원을 호출하고, 중단 시 원응답을 재전송하지 않게 검증한다. 비교 행렬·판정·선택은 이 노트북이 담당한다. HTML은 결과를 직접 확인한 뒤 작성한다.

셀은 추출→검색→원문 근거 검토→답변→평가로 나눠 실행한다. 아래 준비 셀은 67번 선택이 확정되기 전에는 실행하지 않는다.

In [13]:
from pathlib import Path
import importlib.util, json, os, time, resource, copy
import numpy as np
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
def module(name,filename):
    spec=importlib.util.spec_from_file_location(name,ROOT/'notebooks/scripts'/filename)
    value=importlib.util.module_from_spec(spec);spec.loader.exec_module(value);return value
V=module('retrieval68_notebook','68_69_retrieval_runtime.py');T=V.T;A=V.A
F=module('facts68_notebook','68_fact_input.py')
BASE=T.R.P.F.STAGES[68];BASE.mkdir(exist_ok=True)
selected=T.read(ROOT/'notebooks/data/67_structured_retrieval_repair/selection.json')
assert selected['status']=='accepted_development_search_candidate_for68'
T.E.verify_file_map(selected['files_sha256'],ROOT)
qids=('R01','R02','R04','R11','R14','R25','R29','R32','R42','R45')
queries65=T.read(T.R.B65/'questions.json')['queries'];byid={q['query_id']:q for q in queries65}
historical=T.read(ROOT/'notebooks/data/53_sequential_fact_supplement/questions.json')['queries']
queries=[copy.deepcopy(byid[qid])for qid in qids]+[dict(q,group='explicit')for q in historical if q['query_id']in('H08','H11')]
assert len(queries)==12
T.R.X.frozen(BASE/'questions.json',dict(queries=queries,split='error_focused_development',selection_before_answers=True))
old_contract=T.read(ROOT/'notebooks/data/66_visible_answer_policy/question_contract.json')
rubrics=[copy.deepcopy(r)for r in old_contract['records']if r['query_id']in qids]
for qid,descriptions in [('H08',['그린카드 v2의 커피 월 할인 횟수를 설명한다.','최소 전월실적을 설명한다.','월 통합 할인한도를 설명한다.']),
    ('H11',['동일 카드의 넷플릭스와 유튜브 프리미엄 각각20%이상 할인 경로를 추천한다.','추천 혜택의 최소 결제금액과 월 횟수 제한을 설명하며 확인되지 않으면 그 범위를 명시한다.'])]:
    q=next(q for q in queries if q['query_id']==qid)
    rubrics.append(dict(query_id=qid,question=q['question'],question_type='detail'if qid=='H08'else'recommendation_detail',group='explicit',
        requirements=[dict(id=f'{qid}.D{i}',description=d)for i,d in enumerate(descriptions,1)],policy_notes='직접요구와 화면의 실제 주장을 평가한다. 미요청 설명 생략만으로 실패시키지 않는다.'))
T.R.X.frozen(BASE/'question_contract.json',dict(status='frozen_before68_answers',records=rubrics,independent_human_gold=False,
    inherited_policy=T.E.POLICY_VERSION,source66_contract_sha256=T.E.file_digest(ROOT/'notebooks/data/66_visible_answer_policy/question_contract.json')))
# 원질문과 모델이 같은 과거 원응답에서 query embedding을 재사용한다.
raw49=ROOT/'notebooks/data/49_selected_pipeline_confirmation/api_cache/embedding_batch_7e55bb0ee3a0a925fa338b673b179b55eaf7e60c5bec65105780fc0e4b21300a.json'
raw65files=list((T.R.B65/'api_cache').glob('embedding_batch_*.json'))
vector_sources=[(p,T.read(p))for p in [*raw65files,raw49]]
vector_rows=[];receipts=[]
for q in queries:
    matches=[]
    for path,raw in vector_sources:
        request=raw['request'];texts=request['input']
        if q['question']in texts:
            index=texts.index(q['question']);item=next(v for v in raw['raw_response']['data']if v['index']==index)
            assert request['model']=='text-embedding-3-small'and raw['status']=='ok'and len(item['embedding'])==1536
            matches.append((path,index,item['embedding']))
    assert matches,'정확히 같은 원질문 embedding이 필요합니다.'
    assert all(np.array_equal(np.asarray(matches[0][2],dtype=np.float32),np.asarray(m[2],dtype=np.float32))for m in matches), '동일 원질문 벡터가 캐시 사이에서 달라짐'
    path,index,vector=matches[0];vector_rows.append(vector)
    receipts.append(dict(query_id=q['query_id'],question_sha256=T.R.P.sha(q['question']),raw_path=str(path.relative_to(ROOT)),
        raw_sha256=T.E.file_digest(path),raw_index=index,vector_sha256=T.R.P.sha(np.asarray(vector,dtype=np.float32).tobytes())))
vectors=np.asarray(vector_rows,dtype=np.float32)
T.R.X.frozen(BASE/'embedding_reuse_contract.json',dict(records=receipts,new_embedding_requests=0,model='text-embedding-3-small',dimensions=1536))
if (BASE/'query_vectors.npy').exists():assert np.array_equal(np.load(BASE/'query_vectors.npy',allow_pickle=False),vectors)
else:np.save(BASE/'query_vectors.npy',vectors)
print('질문12와 직접요구 고정, 신규 임베딩0')

질문12와 직접요구 고정, 신규 임베딩0


## 추가 질문 추출과 검색

H08/H11만 선택된 추출기로 새로 세 번 추출한다. 나머지 10개는 67번에서 같은 원질문으로 신규 실행한 선택 추출 결과를 사용한다. 모든 비교 조건에서 같은 반복의 검색 결과를 공통으로 쓴다. 답변은 반복 간 재사용하지 않는다.

다음 셀은 67번 선택 후 동결한 manifest·원본 파일 해시를 만들고, 추가 추출과 검색을 수행하도록 이어 작성한다. 아직 검색·답변은 실행되지 않았다.

In [13]:
source_paths=set(T.R.M.source_hashes())
source_paths.update('notebooks/scripts/'+name for name in ('67_69_execution_runtime.py','67_query_repair.py','67_db_repair.py',
    '68_fact_input.py','68_69_answer_runtime.py','68_69_retrieval_runtime.py','68_test_fact_input.py','68_test_answer_runtime.py','68_test_retrieval_runtime.py','65_scope_repair.py'))
sources={name:T.E.file_digest(ROOT/name)for name in sorted(source_paths)}
critical={**selected['files_sha256']}
for path in [ROOT/'notebooks/data/67_structured_retrieval_repair/selection.json',BASE/'questions.json',BASE/'question_contract.json',
    BASE/'query_vectors.npy',BASE/'embedding_reuse_contract.json']:
    critical[str(path.relative_to(ROOT))]=T.E.file_digest(path)
for receipt in receipts:critical[receipt['raw_path']]=receipt['raw_sha256']
phase={'notebooks/68_structured_evidence_generation.ipynb':dict(prefix='exp68-exec-',sha256=T.phase_code_sha(ROOT/'notebooks/68_structured_evidence_generation.ipynb','exp68-exec-'))}
extra=[q for q in queries if q['query_id'].startswith('H')]
logical=[f"{q['query_id']}:parser:r{r}"for q in extra for r in(1,2,3)]
physical=[T.attempt_identity(i,a)for i in logical for a in(1,2)];extra_by_id={q['query_id']:q for q in extra}
parser_variant=selected['parser_variant'];parser_prompt=T.OLD_PARSER_PROMPT if parser_variant=='old'else T.NEW_PARSER_PROMPT
parser_manifest=dict(stage=680,parser_variant=parser_variant,model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,
    source_sha256=sources,critical_data_sha256=critical,notebook_phase_code_sha256=phase,
    allowed_identities=dict(parser=physical,answer=[]),logical_identity_by_physical={T.attempt_identity(i,a):i for i in logical for a in(1,2)},
    allowed_request_by_identity={i:T.R.P.sha([dict(role='user',content=extra_by_id[i.split(':')[0]]['question'])])for i in physical},
    parser_instructions_sha256=T.R.P.sha(parser_prompt),parser_schema_sha256=T.R.P.sha(T.R.Q.QueryPlan.model_json_schema()),
    logical_parser_requests=6,maximum_parser_requests=12,maximum_answer_requests=0,maximum_physical_requests=12,stage_cost_cap_usd='8')
T.R.X.frozen(T.R.P.F.STAGES[680]/'execution_manifest.json',parser_manifest)
allow_network=os.environ.get('ALLOW_NETWORK')=='1'
parsers={}
for q in queries:
    for repeat in(1,2,3):
        if q['query_id'].startswith('H'):row=T.extract_query(680,q,repeat,allow_network=allow_network)
        else:
            source_base=T.R.P.F.STAGES[670 if parser_variant=='old'else 671]
            row=T.read(source_base/'plans'/f"{q['query_id']}:parser:r{repeat}.json")
            T.validate_parser_checkpoint(row,q,T.read(source_base/'execution_manifest.json'),source_base)
        parsers[q['query_id'],repeat]=row
T.R.X.frozen(BASE/'parser_results.json',dict(records=list(parsers.values()),selected_parser=parser_variant))
for path in [BASE/'parser_results.json']:
    critical[str(path.relative_to(ROOT))]=T.E.file_digest(path)
identities=[f"{q['query_id']}:{profile}:{arm}_{prompt}:r{repeat}"for q in queries for profile in T.R.PROFILES
    for arm in('A','B','C')for prompt in('old','direct')for repeat in(1,2,3)]
prompts=T.R.G.C.operating_prompts()
manifest=dict(stage=68,model='gpt-6-luna',reasoning='medium',source_sha256=sources,critical_data_sha256=critical,
    notebook_phase_code_sha256=phase,notebook_code_sha256={},data_sha256=T.read(T.R.B65/'execution_manifest.json')['data_sha256'],
    allowed_identities=dict(parser=[],answer=identities),maximum_parser_requests=0,maximum_answer_requests=864,
    maximum_embedding_requests=0,maximum_physical_requests=864,stage_cost_cap_usd='8',logical_answers=432,
    prompt_variant_by_identity={i:i.split(':')[2].split('_',1)[1]for i in identities},
    answer_instructions_sha256_by_identity={i:[T.R.P.sha(T.transform_prompt(p,i.split(':')[2].split('_',1)[1]))for p in prompts]for i in identities},
    original_U_preserved=True,manual_gold_in_assembler=False,maximum_payload_bytes=64000,profile_retuning=False)
T.R.X.frozen(BASE/'execution_manifest.json',manifest)
T.R.X.frozen(BASE/'source_inventory.json',V.inventory())
print('추가 추출과 실행계약 동결, 답변 생성은 아직 실행하지 않음')

추가 추출과 실행계약 동결, 답변 생성은 아직 실행하지 않음


### 자원 기록 호환 및 재개 안내

최초 실행은 검색72개·입력432개를 모두 저장한 뒤, 공통 자원 함수와 노트북에 같은 `peak_rss_kib` 키가 중복돼 자원 파일 저장에서 중단됐다. 아래 셀은 실험용 검색 클래스의 **자원 반환값에서만** 중복 키를 제거한다. 운영 코드·검색·근거·모델 요청은 바꾸지 않는다. 재현 시 이 셀을 먼저 실행하고 다음 검색 셀을 실행한다. 원래 동결한 실행 코드와 첫 실패 기록은 보존한다.

다음 실행은 저장된 검색 결과를 검증해 재사용한다. 복구 구간의 자원 사용량을 최초 검색 구간으로 해석하지 않는다. 최초 검색 전체의 CPU·피크 메모리는 미측정이며 검색 행별 저장 시간·API 원응답 시간은 별도로 남아 있다.

In [10]:
# Only experiment-class resource metadata; no retrieval/provider method changes.
_before_resource68=getattr(V.Runtime.resources,'_before_resource68_fix',V.Runtime.resources)
def _resource_metadata68(self):
    values=dict(_before_resource68(self))
    values.pop('peak_rss_kib',None)
    return values
_resource_metadata68._before_resource68_fix=_before_resource68
V.Runtime.resources=_resource_metadata68
print('자원 반환값의 중복 키만 조정; 동결한 검색·입력 코드 유지')

자원 반환값의 중복 키만 조정; 동결한 검색·입력 코드 유지


In [10]:
documents=[json.loads(line)for line in(ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4/documents.jsonl').read_text().splitlines()]
directory=ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4';facts=T.R.R.fact_map(directory)
runtime=V.Runtime(68);searches=[];jobs=[];packages=[];start=time.perf_counter();cpu=time.process_time()
for index,(q,vector)in enumerate(zip(queries,vectors,strict=True)):
    original=T.R.S.original_scope(q['question'],documents)
    for repeat in(1,2,3):
        parser=parsers[q['query_id'],repeat]
        routing=T.K.route(T.R,parser['plan'],original,directory,facts,matcher=selected['matcher'],parser_failed=parser['status']!='ok')
        for profile in(T.R.PROFILES if(index+repeat)%2 else tuple(reversed(T.R.PROFILES))):
            row=V.retrieve(runtime,q,vector,profile,repeat,routing,original,parser_binding=T.R.P.sha(parser))
            searches.append(row);package=F.assemble(q['question'],row['result'],routing,facts)
            packages.append(dict(identity=row['identity'],query_id=q['query_id'],profile=profile,repeat=repeat,
                original_U_result_sha256=T.R.P.sha(row['result']),**{k:v for k,v in package.items()if k!='results'}))
            combinations=[(arm,prompt)for arm in('A','B','C')for prompt in('old','direct')]
            shift=(index+repeat)%6;combinations=combinations[shift:]+combinations[:shift]
            for arm,prompt in combinations:
                identity=f"{q['query_id']}:{profile}:{arm}_{prompt}:r{repeat}"
                jobs.append(dict(identity=identity,query_id=q['query_id'],question=q['question'],profile=profile,
                    mode=f'{arm}_{prompt}',repeat=repeat,result=package['results'][arm],retrieval_status=row['retrieval_status'],
                    fixed_search_identity=row['identity'],package_applicable=package['applicable']))
    print('검색',index+1,'/12',flush=True)
assert len(jobs)==432 and len({j['identity']for j in jobs})==432
T.R.X.frozen(BASE/'retrieval_results.json',dict(records=searches))
T.R.X.frozen(BASE/'packages.json',dict(records=packages,all_B_C_canonical_equal=True,no_U_eviction=True))
T.R.X.frozen(BASE/'jobs.json',dict(records=jobs))
T.R.X.frozen(BASE/'answer_payload_contract.json',A.payload_contract(jobs,manifest))
T.write(BASE/'retrieval_resources.json',dict(**runtime.resources(),segment_wall_seconds=time.perf_counter()-start,
    segment_cpu_seconds=time.process_time()-cpu,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    logical_searches=len(searches),logical_answers=len(jobs),original_U_union=False,new_embeddings=0))
print('검색과 동일 입력 구성 완료. 원문·실제 입력 검토 후에만 다음 답변 셀 실행.')

검색 1 /12
검색 2 /12
검색 3 /12
검색 4 /12
검색 5 /12
검색 6 /12
검색 7 /12
검색 8 /12
검색 9 /12
검색 10 /12
검색 11 /12
검색 12 /12
검색과 동일 입력 구성 완료. 원문·실제 입력 검토 후에만 다음 답변 셀 실행.


In [12]:
import fitz
from collections import Counter
# Evaluation-only source library. Neither retrieval nor assembler reads this file.
documents=[json.loads(line)for line in(ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4/documents.jsonl').read_text().splitlines()]
registry=[];pdf_files={};blank_pages=[];page_total=0
for document in documents:
    key=document['document_id'];pdf=ROOT/'data/raw'/(key+'.pdf');pdf_sha=T.E.file_digest(pdf)
    assert pdf_sha==document['source_pdf_sha256']
    pdf_files[str(pdf.relative_to(ROOT))]=pdf_sha;page_facts=[]
    with fitz.open(pdf)as source:
        for page in source:
            page_total+=1;text=page.get_text().strip()
            if not text:
                blank_pages.append(dict(card_key=key,page=page.number+1));continue
            page_facts.append(dict(condition_id=f'{key}:pdf_page66.{page.number+1}',benefit_id='source_page',
                reference_unit='pdf_page',source_statement='원문 PDF 페이지 참조. 개별 주장 의미는 별도로 수동 판정한다.',
                source_status='verified',source_support=[dict(path=str(pdf.relative_to(ROOT)),sha256=pdf_sha,page=page.number+1,quote=text)]))
    registry.append(dict(query_id='global_source_reference',card_key=key,source_pdf_sha256=pdf_sha,facts=page_facts))
T.R.X.frozen(BASE/'source_registry.json',dict(status='native_PDF_reference_library_before68_answers',records=registry,
    purpose='모든106카드의 원문 참조. 정답카드 제한이나 자동 사실 조립의 입력이 아니다.',
    source_files_sha256=pdf_files,cards=106,PDF_pages=page_total,blank_native_text_pages=blank_pages,
    reference_ID_namespace='pdf_page66 is the reused evaluation-policy reference format, not a claim this was scored in66',
    independent_human_gold=False,answers_seen=0))
T.R.X.frozen(BASE/'source_proof_inventory.json',dict(status='PDF_hash_and_native_page_quotes_verified',files_sha256=pdf_files,
    source_registry_raw_sha256=T.E.file_digest(BASE/'source_registry.json'),pages=page_total,
    nonblank_page_references=sum(len(r['facts'])for r in registry),blank_pages=len(blank_pages),
    claim_truth_not_automatically_inferred=True,provider_answers_seen=0))
jobs=T.read(BASE/'jobs.json')['records'];retrievals=T.read(BASE/'retrieval_results.json')['records']
by_search={row['identity']:row for row in retrievals};facts=T.R.R.fact_map(ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4')
assert len(jobs)==432 and A.payload_contract(jobs,manifest)==T.read(BASE/'answer_payload_contract.json')
reassembled={}
for job in jobs:
    sid=job['fixed_search_identity'];row=by_search[sid]
    if sid not in reassembled:
        fresh_route=T.K.route(T.R,row['routing']['plan'],row['original_scope'],ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4',facts,matcher=selected['matcher'])
        assert {k:v for k,v in fresh_route.items()if k!='db_wall_seconds'}=={k:v for k,v in row['routing'].items()if k!='db_wall_seconds'}
        reassembled[sid]=F.assemble(job['question'],row['result'],fresh_route,facts)
    arm=job['mode'].split('_',1)[0]
    assert job['result']==reassembled[sid]['results'][arm]
assert all(row['retrieval_status']=='ok'for row in retrievals)
T.R.X.frozen(BASE/'planned_input_verification.json',dict(status='all432_planned_payloads_and72_canonical_packages_verified',
    jobs_raw_sha256=T.E.file_digest(BASE/'jobs.json'),payload_contract_raw_sha256=T.E.file_digest(BASE/'answer_payload_contract.json'),
    source_registry_raw_sha256=T.E.file_digest(BASE/'source_registry.json'),source_proof_raw_sha256=T.E.file_digest(BASE/'source_proof_inventory.json'),
    original_U_cards_ranks_and_full_text_preserved=True,B_C_same_facts_relations_sources=True,
    all_packages_reassembled_from_DB_and_search_only=True,manual_gold_in_assembler=False,
    logical_jobs=432,retrievals=72,technical_retrieval_success=72,source_pages=page_total,
    actual_provider_transmission_must_be_verified_after_generation=True))
print('원문 참조106카드·',page_total,'쪽과432계획입력 검증. 실제전송검증은 답변생성 후 별도 수행.')

원문 참조106카드· 617 쪽과432계획입력 검증. 실제전송검증은 답변생성 후 별도 수행.


## 답변 생성 전 원문·입력 검토 게이트

선택 카드별 원문 참조, 직접 요청 기준, 원본 비구조화 근거 보존, B/C의 전체 사실·관계 동일성 및 실제 전송할 입력을 검토한다. 이 검토는 정답 근거를 검색 입력에 주입하지 않는다. 명확한 원문 사실과 입력 관계가 없는 경우에는 불확실하다고 기록한다. 검토 수락 파일 없이는 아래 셀을 실행하지 않는다.

In [13]:
# Reload only the accepted disk matrix; bind every call's full job metadata too.
gate=T.read(BASE/'pre_answer_acceptance.json')
assert gate['status']=='accepted_source_and_input_assembly_before_generation'
T.E.verify_file_map(gate['files_sha256'],ROOT)
assert gate['jobs_raw_sha256']==T.E.file_digest(BASE/'jobs.json')
manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
jobs=T.read(BASE/'jobs.json')['records']
assert len(jobs)==432 and len({j['identity']for j in jobs})==432
assert {j['identity']for j in jobs}==set(manifest['allowed_identities']['answer'])
assert all(j['identity']==f"{j['query_id']}:{j['profile']}:{j['mode']}:r{j['repeat']}"for j in jobs)
assert A.payload_contract(jobs,manifest)==T.read(BASE/'answer_payload_contract.json')
node=next(c for c in T.read(ROOT/'notebooks/68_structured_evidence_generation.ipynb')['cells']if c['id']=='exp68-generation-guard-01')
assert T.R.P.sha(''.join(node['source']))==T.read(BASE/'generation_guard_contract.json')['code_sha256']
def _bind_generator68(original,expected_row_hashes,expected_disk_hash):
    def bound(stage,job,*,allow_network=False):
        if(stage!=68 or expected_row_hashes.get(job.get('identity'))!=T.R.P.sha(job)
                or T.E.file_digest(BASE/'jobs.json')!=expected_disk_hash):
            raise RuntimeError('Frozen68 job metadata or disk matrix changed before provider call')
        return original(stage,job,allow_network=allow_network)
    bound._before_job_binding68=original
    return bound
_original_generate68=getattr(A.generate,'_before_job_binding68',A.generate)
A.generate=_bind_generator68(_original_generate68,{j['identity']:T.R.P.sha(j)for j in jobs},gate['jobs_raw_sha256'])
# These probes must be rejected by the wrapper, before provider/cache activity.
_before_guard_raw_count=len(list((BASE/'api_cache').glob('*.json')))
for field,value in [('profile','wrong_profile'),('question','변조된 질문')]:
    altered=copy.deepcopy(jobs[0]);altered[field]=value
    try:A.generate(68,altered,allow_network=False)
    except RuntimeError as error:assert 'Frozen68 job metadata'in str(error)
    else:raise AssertionError('Stale-memory job probe was not rejected')
assert len(list((BASE/'api_cache').glob('*.json')))==_before_guard_raw_count
print('디스크432행·전체metadata·payload 재검증, stale메타/본문 음성검사2개통과, 신규API0')

디스크432행·전체metadata·payload 재검증, stale메타/본문 음성검사2개통과, 신규API0


In [ ]:
gate=T.read(BASE/'pre_answer_acceptance.json')
assert gate['status']=='accepted_source_and_input_assembly_before_generation'
T.E.verify_file_map(gate['files_sha256'],ROOT)
assert gate['jobs_raw_sha256']==T.E.file_digest(BASE/'jobs.json')
rows=[];start=time.perf_counter();cpu=time.process_time()
for index,job in enumerate(jobs,1):
    row=A.generate(68,job,allow_network=allow_network);rows.append(row)
    if index%12==0:
        ledger=T.budget_ledger();print('답변',index,'/432, 누적보수차감',ledger['budget_debit_usd'],flush=True)
T.R.X.frozen(BASE/'execution_results.json',dict(records=rows,logical_answers=432,selection_not_yet_performed=True))
T.write(BASE/'answer_resources.json',dict(segment_wall_seconds=time.perf_counter()-start,segment_cpu_seconds=time.process_time()-cpu,
    peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,ledger=T.budget_ledger(),first_token_time='미측정',browser_display_time='미측정'))
print('432답변 완료. 화면 직접요구·실제주장·입력지원·부가설명을 별도 채점한다.')

## 답변 생성 후 실제 입력·평가 맥락 고정

432답변이 저장된 뒤 실행한다. 계획 입력과 실제 provider 전송 입력을 구분해 재검증한다. 원문 참조는 모든106카드에 공통이며, 특정 정답카드 목록으로 정답을 제한하지 않는다. 질문 요구·실제 화면 주장·실제 입력 지원의 의미 판단은 수동 검토가 담당한다. 해시 검사는 의미 정답을 대신하지 않는다.

In [ ]:
answers=T.read(BASE/'execution_results.json')['records'];jobs=T.read(BASE/'jobs.json')['records']
assert len(answers)==len(jobs)==432
answer_by_id={r['identity']:r for r in answers}
receipts=[];provider_files={}
for job in jobs:
    row=answer_by_id[job['identity']]
    A.validate_answer(row,job,manifest,T.read(BASE/'answer_payload_contract.json'),BASE)
    receipt=T.E.verify_saved_input(job,row,BASE)
    receipts.append(receipt);provider_files.update(receipt['files_sha256'])
T.R.X.frozen(BASE/'input_verification.json',dict(status='all432_actual_saved_provider_inputs_verified',
    records=receipts,files_sha256=provider_files,all_modes_and_repetitions_present=True))
roles=T.E.expected_roles(68)
files={name:T.E.file_digest(ROOT/name)for name in roles.values()}
files.update(provider_files)
files.update(T.read(BASE/'pre_answer_acceptance.json')['files_sha256'])
files.update(T.read(BASE/'source_proof_inventory.json')['files_sha256'])
for path in [BASE/'execution_manifest.json',BASE/'jobs.json',BASE/'execution_results.json',BASE/'pre_answer_acceptance.json',
    BASE/'planned_input_verification.json']:
    files[str(path.relative_to(ROOT))]=T.E.file_digest(path)
source_catalog=T.read(BASE/'source_registry.json')
source_refs={(card['card_key'],fact['condition_id'])for card in source_catalog['records']for fact in card['facts']}
context=dict(status='frozen_before68_semantic_reviews',stage=68,policy_version=T.E.POLICY_VERSION,
    artifact_roles=roles,files_sha256=files,
    source_refs_sha256_by_query={q['query_id']:T.E.digest(sorted(source_refs))for q in queries},
    input_receipts_by_identity={r['identity']:r for r in receipts},manual_semantics_not_inferred_by_hashes=True,
    source_reference_scope='all106 product PDFs; direct benefit fit is manually judged, not a fixed gold-card membership check')
T.R.X.frozen(BASE/'evaluation_context.json',context)
print('실제432입력과 평가정책·질문·원문·요청 계약 결속 완료. 수동 의미 판정을 시작한다.')